# E1.2 · Building the AI and agent inventory

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.1 · Why point-in-time control testing fails for AI](https://spbreed.github.io/cyber-commons/lessons/E1.1.html)**.

| | |
|---|---|
| Tools used | agentgateway, SPIRE |

## What this lesson is

**What it covers.** Discover agents from gateway and identity telemetry; build the register.

**Why a security engineer needs it.** Shadow AI and shadow agents — the inventory is the control most orgs still lack. The control it builds is: discovery, registration, ownership, risk tiering.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Nobody can govern what nobody has listed. The inventory is the least interesting artefact in this function and the one everything else depends on — and the hard part is not building it, it is keeping it true next quarter.

> **At CyberTravels.** Nobody at CyberTravels can currently list every agent, MCP server and vector index in the estate. Everything else in this chapter depends on that list being true next quarter.

## 2 · The framework

```
   what has to be in the register

   model . agent . integration . MCP server . eval corpus
        |
   +----v-------------------------------------------+
   | owner . purpose . data classes . autonomy level|
   | tools it holds . environment . last verified   |
   +------------------------------------------------+

   building it is a project. keeping it true is the control.
```

You cannot govern, tier, test or revoke what you cannot list. The AI inventory is
therefore the first control, not a documentation exercise.

The honest finding of every first inventory is the same: **most of it was already
in production.** Not because anyone was reckless, but because AI features arrive
inside products you already bought, and agents get created programmatically by
other agents.

Three sources, and the third finds what the first two miss:

1. the **model registry** — what your ML team registered,
2. **procurement and expense** — what someone bought,
3. **egress logs to model-provider domains** — what is actually being used.

Source 3 is the one that discovers the department using a frontier API on a
personal card, and the SaaS product that quietly added an AI feature.

## 3 · Resolving one row into a deployment, as a skill

Discovery finds that a thing exists. Governing it needs the row resolved into artefacts: which repository at which commit, which image **digest** rather than which tag, which IAM role and SPIFFE ID, which gateway and guardrail, and every downstream its tools call. Every other attestation skill consumes this graph, which is why it runs first. This is the file in this repository:

In [ ]:
# skills/attestation/deployment-inventory-resolver/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: deployment-inventory-resolver
description: >-
  Resolve a deployment_id into the artifacts that make up one agentic
  deployment and build the evidence graph every other attestation skill
  consumes. Use first, before any control is evaluated, or when asked which
  repo, image, role, workload identity, gateway or guardrail a deployment
  actually consists of.
allowed-tools: Bash, Read, Grep, Glob
---

# Deployment Inventory Resolver

**Controls:** All — this skill produces the join key

## Why this runs first

`deployment_id` is the primary key for every other skill in this set. Without
it, an IAM finding, a SPIFFE entry and a gateway policy are three unrelated
facts about three things that may or may not be the same system.

This skill turns that ID into a manifest of content-addressed artifacts, so
every later finding resolves to one evaluable unit and cannot be silently
confused with a neighbouring deployment.

## Procedure

1. **Resolve the code.** Repository URL plus the exact commit SHA that was
   built. Not a branch — a branch moves.
2. **Resolve the image.** Registry digest (`sha256:…`), not a tag. Confirm the
   digest matches the revision actually deployed, not the newest build.
3. **Resolve the runtime identity.** The IAM role ARN, the SPIFFE ID, and —
   on platforms that auto-create one — the workload identity ARN exposed by the
   runtime/gateway description API.
4. **Resolve the traffic path.** Gateway or route ARN, and the guardrail ID
   attached to it.
5. **Resolve the downstreams.** Every service the deployment's tools call.
   These become the input to the risk-registry skill.
6. **Cross-link and check completeness.** Every artifact must reference the
   others. Report orphans rather than omitting them.

## Output contract

```json
{
  "deployment_id": "str",
  "resolved_at": "str",
  "artifacts": {
    "repo": {"url": "str", "commit": "str"},
    "image": {"registry": "str", "digest": "str", "matches_deployed": true},
    "identity": {"role_arn": "str", "spiffe_id": "str", "workload_identity_arn": "str"},
    "traffic": {"gateway_arn": "str", "guardrail_id": "str"},
    "downstreams": ["str"]
  },
  "missing": ["str"],
  "orphans": ["str"],
  "verdict": "PASS|PARTIAL|FAIL"
}
```

A manifest with entries in `missing` is `PARTIAL` at best. Every downstream
skill inherits that ceiling — you cannot attest a control on an artifact you
could not resolve.

## Failure modes

- **Resolving a tag instead of a digest.** Tags are mutable; the thing you
  attested is not necessarily the thing running.
- **Treating an unresolvable artifact as absent.** "No gateway configured" and
  "I could not read the gateway API" are different findings.
- **Reusing a manifest across runs.** Re-resolve. Drift is the point.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/attestation/deployment-inventory-resolver/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/attestation/deployment-inventory-resolver/scripts/deployment_inventory_resolver.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Build an AI inventory from three sources and measure ownership coverage against what the third one finds.

This is the executable half of the `deployment-inventory-resolver` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass, field

@dataclass
class AIAsset:
    name: str
    kind: str              # model | agent | copilot | embedded-feature
    owner: str = ""
    autonomy: str = "L1"
    data: tuple = ()
    external: bool = False
    discovered_via: str = "registry"

    def gaps(self):
        g = []
        if not self.owner:
            g.append("no named owner — nobody can accept the risk or recertify it")
        if self.discovered_via != "registry":
            g.append(f"not registered — found via {self.discovered_via}")
        if self.autonomy in ("L2.5", "L3") and not self.owner:
            g.append("acts semi-autonomously with nobody accountable")
        return g

REGISTRY = [
 AIAsset("fraud-scoring-model", "model", "risk-eng", "L1", ("customer",)),
 AIAsset("support-summariser", "copilot", "support-eng", "L1", ("customer",)),
]
PROCUREMENT = [
 AIAsset("vendor-contract-analyser", "embedded-feature", "", "L1", ("regulated",),
         discovered_via="expense report"),
]
EGRESS = [
 AIAsset("unknown-openai-usage-marketing", "copilot", "", "L1", ("public",),
         discovered_via="egress logs"),
 AIAsset("pr-remediation-agent", "agent", "", "L2.5", ("customer",), True,
         discovered_via="egress logs"),
 AIAsset("agent-worker-7f3c", "agent", "", "L2.5", ("customer",),
         discovered_via="egress logs"),
]
ALL = REGISTRY + PROCUREMENT + EGRESS
print(f"{'asset':34s}{'kind':18s}{'autonomy':10s}{'owner':14s}found via")
print("-" * 92)
for a in ALL:
    print(f"{a.name:34s}{a.kind:18s}{a.autonomy:10s}{a.owner or '—':14s}{a.discovered_via}")
print(f"\nregistry found {len(REGISTRY)}; the other two sources found "
      f"{len(ALL)-len(REGISTRY)} more.")

unowned = [a for a in ALL if not a.owner]
unregistered = [a for a in ALL if a.discovered_via != "registry"]
high_autonomy_unowned = [a for a in ALL if a.autonomy in ("L2.5","L3") and not a.owner]

print(f"assets                    {len(ALL)}")
print(f"no named owner            {len(unowned)}  {[a.name for a in unowned]}")
print(f"never registered          {len(unregistered)}")
print(f"L2.5+ with no owner       {len(high_autonomy_unowned)}  "
      f"{[a.name for a in high_autonomy_unowned]}")

print("\ngaps in detail:")
for a in ALL:
    for g in a.gaps():
        print(f"   {a.name:34s}{g}")
assert high_autonomy_unowned

MODEL_PROVIDER_DOMAINS = {"api.openai.com", "api.anthropic.com",
                          "generativelanguage.googleapis.com",
                          "api.mistral.ai", "api.together.xyz"}

EGRESS_LOG = [
 {"src": "marketing-workstation-14", "host": "api.openai.com", "bytes": 240_000},
 {"src": "svc-pr-remediation",       "host": "api.anthropic.com", "bytes": 8_400_000},
 {"src": "build-runner-3",           "host": "registry.npmjs.org", "bytes": 90_000},
 {"src": "agent-worker-7f3c",        "host": "api.together.xyz", "bytes": 1_200_000},
]
def discover(log, known_names):
    found = []
    for row in log:
        if row["host"] not in MODEL_PROVIDER_DOMAINS: continue
        if row["src"] in known_names: continue
        found.append({"source": row["src"], "provider": row["host"],
                      "volume": row["bytes"],
                      "finding": "AI usage not present in the inventory"})
    return found

known = {a.name for a in REGISTRY + PROCUREMENT}
for f in discover(EGRESS_LOG, known):
    print(f"{f['source']:30s}{f['provider']:34s}{f['volume']:>10,} bytes")
    print(f"{'':30s}{f['finding']}")

def inventory_health(assets):
    return {"total": len(assets),
            "owned": sum(1 for a in assets if a.owner),
            "registered": sum(1 for a in assets if a.discovered_via == "registry"),
            "coverage": round(sum(1 for a in assets if a.owner)/len(assets), 2)}
print(f"\n{inventory_health(ALL)}")

## What you just proved

The skill loads and reports its shape. Two of its rules are what make an inventory hold: resolve digests rather than tags, because a tag is mutable and the thing you attested is not the thing running; and never record an unresolvable artefact as absent — "no gateway configured" and "could not read the gateway" are different findings with different owners.

## Your turn

Run the egress query for real: one week of traffic to model-provider domains, joined against your inventory. It takes an hour and it always finds something.

---

**Next → [E1.3 · Risk tiering agentic use cases](https://spbreed.github.io/cyber-commons/lessons/E1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*